In [ ]:
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')

In [ ]:
!pip install biopython

In [ ]:
import re
import pandas as pd
from google.colab import files

print("Please select your .txt file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

records = []

with open(file_name, 'r', encoding='utf-8', errors='ignore') as f:
    text = f.read()

entries = text.strip().split('\n\n')

for entry in entries:
    seq_match = re.search(r'(?:SeqID|Sequence Name):\s*(\S+)', entry)
    loc_match = re.search(r'Final Prediction:\s*([^\n]+)', entry)

    if seq_match and loc_match:
        records.append({
            'SeqID': seq_match.group(1),
            'Localization': loc_match.group(1).strip()
        })

if not records:
    try:
        df = pd.read_csv(file_name, sep='\t')
        cols = [c for c in df.columns if c in ['SeqID', 'Localization', 'Final Prediction']]
        if cols:
            df = df[cols]
    except Exception:
        print("Could not parse file as a table. Please check the PSORTb format.")
else:
    df = pd.DataFrame(records)

print("\n--- First 5 Proteins ---")
display(df.head())

print("\n--- Subcellular Localization Distribution ---")
summary = df['Localization'].value_counts().reset_index()
summary.columns = ['Localization', 'Protein Count']
display(summary)

output_name = 'PSORTb_Parsed_Results.xlsx'
with pd.ExcelWriter(output_name) as writer:
    df.to_excel(writer, sheet_name='All_Proteins', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)

files.download(output_name)

In [ ]:
import pandas as pd

df=pd.read_excel("/content/PSORTb_Parsed_Results.xlsx")
df.head(10)

In [ ]:

ready_dataset = df[df['Localization'] != 'Unknown']
ready_dataset.head(20)

In [ ]:
df['Localization_Clean'] = df['Localization'].str.extract(r'([A-Za-z]+)')

In [ ]:
drug_candidates = df[df['Localization_Clean'] == 'Cytoplasmic']
drug_candidates.head(20)

In [ ]:
import pandas as pd
scores = df['Localization'].str.extract(r'(\d+\.\d+|\d+)')[0]
df['Score'] = pd.to_numeric(scores, errors='coerce')


In [ ]:
drug_candidates[drug_candidates['Score']>7.50]

In [ ]:
from Bio import SeqIO

input_fasta = "/content/drive/MyDrive/Subtractive Genomics Approach/m_tuberculosis_EDG_filter.fasta"
output_fasta = "/content/drive/MyDrive/Subtractive Genomics Approach/top_candidates_score_10.fasta"


seq_dict = SeqIO.to_dict(SeqIO.parse(input_fasta, "fasta"))

selected_records = (
    drug_candidates.loc[drug_candidates['Score'] == 10, 'SeqID']
    .map(seq_dict)
    .dropna()
    .tolist()
)

SeqIO.write(selected_records, output_fasta, "fasta")